In [ ]:
import Affichage
import Data_Generation
import Heuristic
import Pretraitement
import Tabu_Search
import PL
import Score
import time
import networkx as nx

instance = [
    "Bobillot,Montreuil,FRANCE",
    "Centre ville,Montreuil,FRANCE",
    "Barbusse,Montreuil,FRANCE",
    "Ruffins,Montreuil,FRANCE",
    "Centre-ville,Vincennes,FRANCE",
    "République,Vincennes,FRANCE",
    "Diderot,Vincennes,FRANCE",
    "Vignerons,Vincennes,FRANCE",
    "Domaine du bois,Vincennes,FRANCE",
    "Yvoire,FRANCE",
    "Houilles,France"
]

path = instance[0]

## Partie I - Prétraitement

In [ ]:
Graphe = Data_Generation.import_instance("Instances/"+path)

A,P,E,V = Data_Generation.path_and_edges(Graphe)

**A** correspond à l'ensemble des arcs et aretes du graphe de la forme (u,v,w) avec w le poid des arcs  
**P** correspond à l'ensemble des couples st différents  
**E** est l'ensemble des indices des arcs à doubles sens dans A tel que si on a A[i] = (u,v,w) et A[j] = (v,u,w) alors E = [...,i,j,...]  
**V** est l'ensemble des sommets du Graphe  

In [ ]:
P_candidate = Pretraitement.candidat_st(Graphe,A,E)

Strong_Bridge = Pretraitement.Strong_Bridges(Graphe,A,E)

A_update, E_update, Graphe_update = Pretraitement.Update_Graphe(A,E,Strong_Bridge)

**P_candidate** correspond aux couples st résultant du pretraitement  
**Strong_Bridge** est un liste d'arc (u,v) qui sont des strong bridge du graphe  
**A_update** et **E_update** sont l'update de A et E après avoir enlevé les arcs inverses des strong bridges dans le graphe  

In [ ]:
print("|A| = ",len(A))
print("|P| = ",len(P))
print("|E| = ",len(E))
print("|V| = ",len(V))

In [ ]:
print("|A'| = ", len(A_update))
print("|P'| = ", len(P_candidate))
print("Nombre de Strong Bridge: ", len(Strong_Bridge))
print("|E'| = ", len(E_update))


## Partie II - PL

In [ ]:
#Calcule de la solution optimale du PL sans prétraitement
solution, val_PL, t = PL.PL(A,P_candidate,E,V)

value = Score.compute_value(A,E,solution,P) #Calcul du score de la solution sans pretraitement

In [ ]:
#Calcule de la solution optimale du PL avec prétraitement
solution_p, val_PL_p, t_p = PL.PL(A_update,P_candidate,E_update,V)

value_p =  Score.compute_value(A_update,E_update,solution_p,P) #Calcul du score de la solution avec pretraitement

In [ ]:
print("Valeur de la solution optimal: ", value)
print("Temps d'excution du PL sans pretraitement: ", round(t,3), "s")

In [ ]:
print("Valeur de la solution optimal: ", value_p)
print("Temps d'excution du PL sans pretraitement: ", round(t_p,3), "s")

## Partie III - Tabu Search

In [ ]:
#Parametre de l'algorithme
it = 1000
maxTabuSize = 7
seed = 0

In [ ]:
# Génération de la solution de initial aléatoire

s0 = Heuristic.heuristic(Graphe_update,A_update,E_update,seed)


In [ ]:
# Algorithme Tabu

t_start = time.time()
sBest,n = Tabu_Search.tabu(s0, it, maxTabuSize, Tabu_Search.fitness, Tabu_Search.getNeighbors, Tabu_Search.tabuTest, A_update,E_update)
t_end = time.time()

value_tabu = Score.compute_value(A_update,E_update,sBest,P)

In [ ]:
print("Valeur de la solution optimal: ", value_p)
print("Nombre d'itération: ", n)
print("Temps d'excution du Tabu Search: ", round(t_end-t_start,3), "s")

## Partie IV - Visualisation

### 1) Affichage du graphe de la ville

In [ ]:
Affichage.print_graphe(Graphe,path)

### 2) Affichage du graphe de la ville avec les aretes à orienter

In [ ]:
Affichage.print_graphe_arete(Graphe,path,A,E)

### 3) Affichage de la ville avec les Strong Bridges en vert et les aretes à orienter en rouge

In [ ]:
Affichage.print_graphe_Strong_Bridges(Graphe,path,A,E,Strong_Bridge)